In [19]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

import pickle

from sklearn.metrics import root_mean_squared_error

In [20]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/cszczepanek/projects/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1791979857917, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791979857917, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [21]:
def read_parquet(file_link):
    df = pd.read_parquet(file_link)

    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[df.duration >= 1]
    df = df[df.duration <= 60]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [22]:
df_train = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_eval = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [23]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_eval['PU_DO'] = df_eval['PULocationID'] + '_' + df_eval['DOLocationID']

In [24]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

eval_dicts = df_eval[categorical + numerical].to_dict(orient='records')
X_eval = dv.transform(eval_dicts)

In [25]:
import numpy as np

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_eval.indices = X_eval.indices.astype(np.int32)
X_eval.indptr = X_eval.indptr.astype(np.int32)

In [26]:
target = 'duration'
y_train = df_train[target].values
y_eval = df_eval[target].values

In [27]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.7587152100640475

In [28]:
with open('../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [46]:
with mlflow.start_run():

    mlflow.set_tag("developer", "Cezary")

    mlflow.log_param("train-data-link", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-link", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")
    
    alpha=0.1
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_eval)
    
    rmse = root_mean_squared_error(y_eval, y_pred)
    mlflow.log_metric("RMSE", rmse)

    mlflow.log_artifact(local_path="../models/lin_reg.bin", artifact_path="models_pickle")

In [30]:
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.681269743902628

In [39]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [40]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_eval, label=y_eval)

In [41]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_eval, y_pred)
        mlflow.log_metric("RMSE", rmse)
    return {"loss" : rmse, "status" : STATUS_OK}

In [42]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:11.45290                                                   
[1]	validation-rmse:10.78719                                                   
[2]	validation-rmse:10.20563                                                   
  0%|                                   | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/cszczepanek/projects/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:57:25] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:9.69988                                                    
[4]	validation-rmse:9.26119                                                    
[5]	validation-rmse:8.88285                                                    
[6]	validation-rmse:8.55738                                                    
[7]	validation-rmse:8.27809                                                    
[8]	validation-rmse:8.03928                                                    
[9]	validation-rmse:7.83563                                                    
[10]	validation-rmse:7.66216                                                   
[11]	validation-rmse:7.51299                                                   
[12]	validation-rmse:7.38754                                                   
[13]	validation-rmse:7.28102                                                   
[14]	validation-rmse:7.18996                                                   
[15]	validation-rmse:7.11338            

/Users/cszczepanek/projects/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:57:54] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:9.90387                                                    
[1]	validation-rmse:8.48037                                                    
[2]	validation-rmse:7.63770                                                    
[3]	validation-rmse:7.14446                                                    
[4]	validation-rmse:6.86461                                                    
[5]	validation-rmse:6.69873                                                    
[6]	validation-rmse:6.60420                                                    
[7]	validation-rmse:6.54771                                                    
[8]	validation-rmse:6.50674                                                    
[9]	validation-rmse:6.48088                                                    
[10]	validation-rmse:6.46274                                                   
[11]	validation-rmse:6.44656                                                   
[12]	validation-rmse:6.43337            

...

/Users/cszczepanek/projects/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:07:56] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:8.78366                                                    
[1]	validation-rmse:7.41449                                                    
[2]	validation-rmse:6.89938                                                    
[3]	validation-rmse:6.70127                                                    
[4]	validation-rmse:6.61285                                                    
[5]	validation-rmse:6.57109                                                    
[6]	validation-rmse:6.54949                                                    
[7]	validation-rmse:6.53918                                                    
[8]	validation-rmse:6.53154                                                    
[9]	validation-rmse:6.52725                                                    
[10]	validation-rmse:6.52073                                                   
[11]	validation-rmse:6.51519                                                   
[12]	validation-rmse:6.51134            

/Users/cszczepanek/projects/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:08:11] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.73835                                                   
[1]	validation-rmse:11.29968                                                   
[2]	validation-rmse:10.89482                                                   
[3]	validation-rmse:10.52101                                                   
[4]	validation-rmse:10.17804                                                   
[5]	validation-rmse:9.86133                                                    
[6]	validation-rmse:9.57216                                                    
[7]	validation-rmse:9.30554                                                    
[8]	validation-rmse:9.06138                                                    
[9]	validation-rmse:8.83818                                                    
[10]	validation-rmse:8.63445                                                   
[11]	validation-rmse:8.44793                                                   
[12]	validation-rmse:8.27809            

KeyboardInterrupt: 

In [50]:
mlflow.xgboost.autolog(disable=True)

In [52]:
with mlflow.start_run():

    best_params = {
        "learning_rate" : 0.2948646839243271,
        "max_depth" : 30,
        "min_child_weight" : 1.9805346649872448,
        "objective" : "reg:squarederror",
        "reg_alpha" : 0.06275960619157453,
        "reg_lambda" : 0.22388496105060088,
        "seed" : 42
    }

    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_eval, y_pred)
    mlflow.log_metric("RMSE", rmse)

    with open("../models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:9.92391
[1]	validation-rmse:8.52365
[2]	validation-rmse:7.69333
[3]	validation-rmse:7.21505
[4]	validation-rmse:6.93944
[5]	validation-rmse:6.77711
[6]	validation-rmse:6.68196
[7]	validation-rmse:6.62197
[8]	validation-rmse:6.58173
[9]	validation-rmse:6.55463
[10]	validation-rmse:6.53570
[11]	validation-rmse:6.52075
[12]	validation-rmse:6.51208
[13]	validation-rmse:6.50465
[14]	validation-rmse:6.50212
[15]	validation-rmse:6.49959
[16]	validation-rmse:6.49611
[17]	validation-rmse:6.49361
[18]	validation-rmse:6.48940
[19]	validation-rmse:6.48529
[20]	validation-rmse:6.48214
[21]	validation-rmse:6.47965
[22]	validation-rmse:6.47731
[23]	validation-rmse:6.47499
[24]	validation-rmse:6.47183
[25]	validation-rmse:6.46963
[26]	validation-rmse:6.46684
[27]	validation-rmse:6.46545
[28]	validation-rmse:6.46225
[29]	validation-rmse:6.46082
[30]	validation-rmse:6.45909
[31]	validation-rmse:6.45668
[32]	validation-rmse:6.45324
[33]	validation-rmse:6.45159
[34]	validation-rmse:6.4

2026/10/14 16:58:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/14 16:58:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [54]:
logged_model = 'models:/m-27d6a2620cf343fa8faf5bc5848cf85b'

loaded_model = mlflow.pyfunc.load_model(logged_model)

In [57]:
xgboost_model = mlflow.xgboost.load_model(logged_model)
xgboost_model

In [58]:
y_pred = xgboost_model.predict(valid)
y_pred[:10]

array([14.586527 ,  7.2014832, 14.387785 , 24.29462  ,  9.860293 ,
       17.205248 , 11.048476 ,  8.38429  ,  9.754515 , 20.591978 ],
      dtype=float32)